# Dose-dependent sensitivity of human 3D chromatin to a heart disease-linked transcription factor

https://www.biorxiv.org/content/10.1101/2025.01.09.632202v1


# Atrial and ventricular differentiation time course
* 4DNESG6I8TG1 - 4DNFIUMP8ZZ6 - undifferentiated

* 4DNES4UH8UR5 - 4DNFIK5SQFY2 - atrial - 2 days
* 4DNES4T1TMBA - 4DNFIUD4ECSX - atrial - 4 days
* 4DNESW2MTVKJ - 4DNFI4N2S9QW - atrial - 6 days
* 4DNESAMIZR39 - 4DNFI4MXUCUV - atrial - 11 days
* 4DNESR9IH796 - 4DNFI1LYV4VR - atrial - 20 days
* 4DNESCXE1LCX - 4DNFIX5V9GBA - atrial - 45 days

* 4DNES4P4UOAZ - 4DNFIJPZ4ASS - ventricular - 2 days
* 4DNESNRK4ZL7 - 4DNFIM3UJ1XI - ventricular - 4 days
* 4DNES7R8BYB1 - 4DNFIRHM7URR - ventricular - 6 days
* 4DNES7CDN1M9 - 4DNFIYBBPZ1V - ventricular - 11 days
* 4DNESDVZJW63 - 4DNFIQ57GW6C - ventricular - 23 days

* 4DNES4NO6DRP - 4DNFI32RESZT - cardiac 20 day TBX5 het
* 4DNESW7WYVHO - 4DNFI52OLNJ4 - cardiac 20 day TBX5 compound het
* 4DNES9ZDUTHD - 4DNFIHFX73VQ - cardiac 20 day WT
* 4DNESHYMOHEK - 4DNFI5MNOB1Y - Crispr treated, unmodified control WTC-11 derived cardiac myocytes - differentiation day 23
* 4DNES23G2JHL - 4DNFIO73IAII - TBX5 heterozygous WTC-11-derived cardiac myocytes - differentiation day 23
* 4DNES8CZNQCU - 4DNFIL2ODN92 - TBX5 homozygous (compound heterozygous) WTC 11-derived cardiac myocytes - differentiation day 23
* 4DNESNXD7O8Y - 4DNFI22MZ51B - Untreated, unmodified WTC-11-derived cardiac myocytes - differentiation day 23


In [ ]:
from dlem.feature_extraction import extractor
import dlem.util as util
from dlem.loader import load_model, load_reader
from cooltools.lib.numutils import adaptive_coarsegrain, interp_nan

import cooler
import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt
import time
from pathlib import Path
import os
from joblib import Parallel, delayed


In [ ]:
def weighted_mse(input, target, weight=None):
    if weight is None:
        weight = torch.exp(target)
    return torch.mean(weight * (input - target) ** 2)

def wMSE_weight(weight_matrix):
    return weight_matrix
    
def dlem_locus(cool_filename, out_dir, locus, res, cell_line):
    locus_sanitized = locus.replace(':', '_').replace('-', '_')
    file_name = Path(cool_filename).stem
    out_img_path = f'{out_dir}/{file_name}_{locus_sanitized}_{cell_line}.png'
    weights = wMSE_weight
    
    lr = 0.5
    num_epoch = 100
    detach = {'10000':0.025,
              '5000':0.00125,
              '2000':0.005}
    dev_name = "cpu"
    h5 = h5py.File(cool_filename, 'r')
    cool = cooler.Cooler(h5['resolutions'][res])
    
    model_name = 'minimal_dlem'
    
    diag_start= 5 * int(10_000/int(res))
    diag_stop= 120 * int(10_000/int(res))
    depth = 10 * int(10_000/int(res))
    
    res = int(res)
    
    patch_coarsegrain = adaptive_coarsegrain(cool.matrix(balance=True).fetch(locus), cool.matrix(balance=False).fetch(locus), cutoff=3, max_levels=8)
    patch_coarsegrain_interpolated = interp_nan(patch_coarsegrain)               
    patch = np.exp(util.diagonal_normalize(np.log( patch_coarsegrain_interpolated[np.newaxis]) ))
    
    start_time = time.perf_counter()
    out = extractor(util.diagonal_normalize(np.log(patch))[0],
                                    res=res,
                                    learning_rate=lr,
                                    arch=model_name,
                                    diag_start=diag_start,
                                    diag_stop=diag_stop,
                                    depth=depth,
                                    num_epoch = num_epoch,
                                    loss=weighted_mse,
                                    weights=weights(patch),
                                    dev_name=dev_name, 
                                    do_plot=True,
                                    plot_path=out_img_path)

    #column_names = ['chrom', 
    #                'start', 
    #                'end', 
    #                "left", 
    #                "right"]
    #column_types = [str, int, int, float, float]
    #column_init = dict(zip(column_names, column_types))
    #results = pd.DataFrame({col: np.empty(length, dtype=dtype) for col, dtype in column_init.items()})
#
    #results.iloc[pd_indx, 0] = chr_arr
    #results.iloc[pd_indx, 1] = start_arr
    #results.iloc[pd_indx, 2] = end_arr
    #results.iloc[pd_indx, 3] = i
    #results.iloc[pd_indx, 4] = out[-1]
    #results.iloc[pd_indx, 5] = perc_nan
    #results.iloc[pd_indx, 6] = distance * (1-perc_nan)
    #for p_i, param in enumerate(out[0]):
    #    results.iloc[pd_indx, 7 + p_i] = param    
    #    
    #results.to_csv(f'{out_dir}/{file_name}_{locus_sanitized}_{cell_line}.txt', sep='\t', index=False)
    
    return out


In [ ]:
# 4DN mappings and time / gene groups
in_celline_map = {
    "4DNFIUMP8ZZ6" : "undifferentiated",
    "4DNFIK5SQFY2" : "atrial_2_days",
    "4DNFIUD4ECSX" : "atrial_4_days",
    "4DNFI4N2S9QW" : "atrial_6_days",
    "4DNFI4MXUCUV" : "atrial_11_days",
    "4DNFI1LYV4VR" : "atrial_20_days",
    "4DNFIX5V9GBA" : "atrial_45_days",
    "4DNFIJPZ4ASS" : "ventricular_2_days",
    "4DNFIM3UJ1XI" : "ventricular_4_days",
    "4DNFIRHM7URR" : "ventricular_6_days",
    "4DNFIYBBPZ1V" : "ventricular_11_days",
    "4DNFIQ57GW6C" : "ventricular_23_days",
    "4DNFI32RESZT" : "cardiac_20_day_TBX5_het",
    "4DNFI52OLNJ4" : "cardiac_20_day_TBX5_compound_het",
    "4DNFIHFX73VQ" : "cardiac_20_day_WT",
    "4DNFI5MNOB1Y" : "crspr_treat_wt_23_days",
    "4DNFIO73IAII" : "tbx_het_23_days",
    "4DNFIL2ODN92" : "tbx_homo_23_days",
    "4DNFI22MZ51B" : "wtc_23_days"
}

cell_groups = {
    "4DNFIUMP8ZZ6" : "atrial_vent_timecourse",
    "4DNFIK5SQFY2" : "atrial_vent_timecourse",
    "4DNFIUD4ECSX" : "atrial_vent_timecourse",
    "4DNFI4N2S9QW" : "atrial_vent_timecourse",
    "4DNFI4MXUCUV" : "atrial_vent_timecourse",
    "4DNFI1LYV4VR" : "atrial_vent_timecourse",
    "4DNFIX5V9GBA" : "atrial_vent_timecourse",
    "4DNFIJPZ4ASS" : "atrial_vent_timecourse",
    "4DNFIM3UJ1XI" : "atrial_vent_timecourse",
    "4DNFIRHM7URR" : "atrial_vent_timecourse",
    "4DNFIYBBPZ1V" : "atrial_vent_timecourse",
    "4DNFIQ57GW6C" : "atrial_vent_timecourse",
    "4DNFI32RESZT" : "tbx5_gene_dosage",
    "4DNFI52OLNJ4" : "tbx5_gene_dosage",
    "4DNFIHFX73VQ" : "tbx5_gene_dosage",
    "4DNFI5MNOB1Y" : "tbx5_gene_dosage",
    "4DNFIO73IAII" : "tbx5_gene_dosage",
    "4DNFIL2ODN92" : "tbx5_gene_dosage",
    "4DNFI22MZ51B" : "tbx5_gene_dosage"
}
#Defining loci, parameters, and input directory
in_data_dir = '/dresch_data/hd_1/diego_files/seq_data/4dn_cardiac_2025/'
locus_1 = 'chr1:235824124-237942085'
locus_2 = 'chr4:64000000-65000000'
res = '10000'
num_proc = 32

In [ ]:
# Differentiation time course
condition = "atrial_vent_timecourse"
time_samples=[]
for in_filename, cell_line  in in_celline_map.items():
    if cell_groups[in_filename] == condition:
        time_samples.append(in_filename)
locus_sanitized = locus_1.replace(':', '_').replace('-', '_')
out_time_dir = os.path.join(in_data_dir, condition)
if not os.path.exists(out_time_dir):
    os.makedirs(out_time_dir)
        
results = Parallel(n_jobs=num_proc, temp_folder=out_time_dir)(
    delayed(dlem_locus)(
        f'{in_data_dir}/{in_filename}.mcool', 
                         out_time_dir,
                         locus_1, 
                         res, 
                         in_celline_map[in_filename]
    )
    for in_filename  in time_samples
)
        

In [ ]:
# TBX5 Gene dosage
condition = "tbx5_gene_dosage"
tbx_samples=[]
for in_filename, cell_line  in in_celline_map.items():
    if cell_groups[in_filename] == condition:
        tbx_samples.append(in_filename)
        
locus_sanitized = locus_2.replace(':', '_').replace('-', '_')
out_tbx_dir = os.path.join(in_data_dir, condition)
if not os.path.exists(out_tbx_dir):
    os.makedirs(out_tbx_dir)
results = Parallel(n_jobs=num_proc, temp_folder=out_tbx_dir)(
    delayed(dlem_locus)(
        f'{in_data_dir}/{in_filename}.mcool', 
                         out_tbx_dir,
                         locus_2, 
                         res, 
                         in_celline_map[in_filename]
    )
    for in_filename  in tbx_samples
)